In [4]:
import pandas as pd
import numpy as np

def generate_realistic_call_data_v2(n_calls=2000, drop_ratio=0.08, seed=42):
    """
    Improved synthetic data generator for call drop prediction.
    Adds: network-side drops, handover failures, ambiguous short calls,
    causal effect of tower load & speed, and realistic signal noise.
    """
    np.random.seed(seed)
    records = []
    call_id = 0

    def get_tower_load(hour, is_drop, network_failure=False):
        # Base load depends on hour
        if (7 <= hour <= 9) or (17 <= hour <= 19):
            base = np.random.randint(70, 95)
        else:
            base = np.random.randint(30, 70)
        # Network failure can cause drop even at low load
        if network_failure:
            base = np.random.randint(20, 60)   # drops can happen at low load too
        # Drops are more likely under high load, but not deterministic
        extra = np.random.randint(-10, 30) if is_drop else np.random.randint(-20, 10)
        load = np.clip(base + extra, 0, 100)
        return load

    for call in range(n_calls):
        # 1. Decide drop type: RF-based (signal), network-based, or handover-based
        drop_type = None
        is_drop = np.random.random() < drop_ratio
        if is_drop:
            # Among drops: 60% RF crash, 20% network failure, 20% handover failure
            r = np.random.random()
            if r < 0.6:
                drop_type = 'rf'
            elif r < 0.8:
                drop_type = 'network'
            else:
                drop_type = 'handover'

        # 2. Call duration (short calls are ambiguous)
        if not is_drop:
            duration = np.random.randint(30, 240)    # normal calls can be long
        else:
            if drop_type == 'network':
                duration = np.random.randint(20, 120)  # network drops often short
            else:
                duration = np.random.randint(10, 90)   # RF/handover drops short

        # 3. Starting signal strength (some drops start poor, some good)
        if drop_type == 'rf':
            start_rsrp = np.random.randint(-95, -70)   # already weaker
        else:
            start_rsrp = np.random.randint(-85, -65)   # stronger

        # 4. Signal evolution
        steps = np.random.randn(duration).cumsum() * 0.8   # more noise
        # Mild degradation for all calls (realistic)
        trend = np.linspace(0, -5, duration)
        if drop_type == 'rf':
            trend = np.linspace(0, -25, duration)        # severe degradation
        elif drop_type == 'network' or drop_type == 'handover':
            trend = np.linspace(0, -5, duration)         # no RF crash
        rsrp_vals = start_rsrp + trend + steps
        rsrp_vals = np.clip(rsrp_vals, -130, -44)

        # 5. Specific RF crash for RF drops
        if drop_type == 'rf':
            crash_start = int(duration * 0.7)
            for t in range(crash_start, duration):
                rsrp_vals[t] = np.minimum(rsrp_vals[t], -110 - (t - crash_start))
            # Ensure final seconds are very low
            rsrp_vals[-5:] = np.linspace(rsrp_vals[-5], -125, 5)

        # 6. Handover drop: simulate signal oscillation before sudden loss
        if drop_type == 'handover':
            # Last 10 seconds: signal spikes down and up then dies
            ho_start = duration - 10
            for t in range(max(0, ho_start), duration):
                if (t - ho_start) % 3 == 0:
                    rsrp_vals[t] -= 15   # sudden dip
                elif (t - ho_start) % 3 == 1:
                    rsrp_vals[t] += 10   # recovery attempt
                else:
                    rsrp_vals[t] -= 20   # final crash
            rsrp_vals = np.clip(rsrp_vals, -130, -44)

        # 7. Ambiguous normal calls: simulate user hanging up in weak signal
        if not is_drop and np.random.random() < 0.15:
            rsrp_vals[-8:] = np.random.randint(-115, -100, size=8)

        # 8. RSRQ and SINR – make them partially independent of RSRP
        base_rsrq = -10 + (rsrp_vals + 90) / 10
        noise_rsrq = np.random.randn(duration) * 3   # more independent noise
        rsrq_vals = base_rsrq + noise_rsrq
        rsrq_vals = np.clip(rsrq_vals, -20, -3)

        base_sinr = 15 + (rsrp_vals + 90) / 8
        noise_sinr = np.random.randn(duration) * 4
        sinr_vals = base_sinr + noise_sinr
        sinr_vals = np.clip(sinr_vals, -5, 30)

        # 9. Rain intensity (affects drops but not deterministically)
        rain = np.random.exponential(scale=1.5) if np.random.random() < 0.3 else 0
        if is_drop and drop_type == 'rf' and rain > 5:
            # Heavy rain exacerbates RF crash
            rsrp_vals[-10:] -= 3

        # 10. Tower load and speed (causal factors)
        start_hour = np.random.randint(0, 24)
        network_failure = (drop_type == 'network')
        tower_load = get_tower_load(start_hour, is_drop, network_failure)

        # Speed: drop more likely driving, but not exclusive
        if is_drop:
            scenario = np.random.choice(['stationary', 'walking', 'driving'], p=[0.3, 0.2, 0.5])
        else:
            scenario = np.random.choice(['stationary', 'walking', 'driving'], p=[0.5, 0.3, 0.2])
        if scenario == 'stationary':
            speed = np.random.uniform(0, 5)
        elif scenario == 'walking':
            speed = np.random.uniform(5, 10)
        else:
            speed = np.random.uniform(30, 120)

        # 11. Band selection (some bands more drop-prone)
        if is_drop and drop_type == 'rf':
            band_weights = [0.5, 0.3, 0.2]   # 5G_n78 more prone
        else:
            band_weights = [0.33, 0.33, 0.34]
        band = np.random.choice(['LTE_B3', 'LTE_B20', '5G_n78'], p=band_weights)

        # 12. Build seconds
        base_ts = 1700000000 + call * 100
        for t in range(duration):
            records.append({
                'timestamp': base_ts + t,
                'rsrp': round(rsrp_vals[t], 1),
                'rsrq': round(rsrq_vals[t], 1),
                'sinr': round(sinr_vals[t], 1),
                'rain_intensity': round(rain, 1),
                'call_id': call_id,
                'band': band,
                'tower_load': tower_load,
                'speed_kmph': round(speed, 1),
                'is_drop': int(is_drop)
            })
        call_id += 1

    return pd.DataFrame(records)

# Generate dataset
df = generate_realistic_call_data_v2(n_calls=2000, drop_ratio=0.08, seed=42)
print(df['is_drop'].value_counts())
print("\nFirst few rows:")
print(df.head())
df.to_csv("../Data/Raw/telecom_call_drop.csv", index=False)
print("\nDataset saved to ../Data/Raw/telecom_call_drop.csv")

is_drop
0    251680
1      7653
Name: count, dtype: int64

First few rows:
    timestamp  rsrp  rsrq  sinr  rain_intensity  call_id    band  tower_load  \
0  1700000000 -70.5 -10.7  18.7             0.0        0  LTE_B3          62   
1  1700000001 -69.3  -7.7  11.1             0.0        0  LTE_B3          62   
2  1700000002 -69.5  -9.4  21.3             0.0        0  LTE_B3          62   
3  1700000003 -69.8  -5.2  22.6             0.0        0  LTE_B3          62   
4  1700000004 -68.5  -3.0  19.2             0.0        0  LTE_B3          62   

   speed_kmph  is_drop  
0         2.3        0  
1         2.3        0  
2         2.3        0  
3         2.3        0  
4         2.3        0  

Dataset saved to ../Data/Raw/telecom_call_drop.csv
is_drop
0    251680
1      7653
Name: count, dtype: int64

First few rows:
    timestamp  rsrp  rsrq  sinr  rain_intensity  call_id    band  tower_load  \
0  1700000000 -70.5 -10.7  18.7             0.0        0  LTE_B3          62   
1  1700